# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ak470107/ML-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Same leakage-safe feature set used everywhere downstream (`w05`/`w06`/`w07`/`capstone`): numeric
signals fed through zero-fills (this dataset's numeric NaNs mean "no data," already handled at
export), plus tier/categorical columns one-hot encoded.

In [1]:
import pandas as pd
import numpy as np
import os

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
elif os.path.exists("ML-internship/data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")
else:
    !git clone --depth 1 https://github.com/ak470107/ML-internship.git
    df = pd.read_csv("ML-internship/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_keyword_data", "has_word_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]

num_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat_frame = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
enc_frame = pd.get_dummies(cat_frame, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([num_frame.reset_index(drop=True), enc_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"  ({len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical, one-hot expanded to {X.shape[1] - len(NUMERIC_FEATURES)} columns)")
print(f"Any NaNs left in X: {X.isna().any().any()}")

Feature matrix: 30,000 rows x 59 columns
  (24 numeric + 8 categorical, one-hot expanded to 35 columns)
Any NaNs left in X: False


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing values | Available before prediction moment? |
|---|---|---|---|
| `search_volume`, `competition`, `cpc` | keyword-level SEO metrics for the page's target term | NaN when no keyword mapped -> zero-filled, flagged via `has_keyword_data` | Yes -- keyword research happens before publish |
| `word_count`, `char_count` | on-page content size | NaN for a few rows -> zero-filled, flagged via `has_word_count` | Yes -- fixed once published |
| `*_90d` traffic/engagement columns | trailing 90-day window, ending at the label's own reference date | 0 = genuinely no traffic (dataset already zero-fills unavailable rows) | Yes -- 90d window is historical relative to the label |
| `content_age_days`, `days_since_last_update` | page lifecycle timing | none | Yes -- structural/CMS fields |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | derived rate metrics over the same 90d window | none (already computed) | Yes -- same window as the 90d columns above |
| tier columns (`freshness_tier`, `word_count_tier`, etc.) | bucketed versions of the above, categorical | `"unknown"` fill for any gaps | Yes |

**The one to watch:** every feature above is a **90-day** rolled-up window, deliberately kept
separate from the two 30-day windows (`*_last_30d`, `*_prev_30d`) that build the label itself --
tested directly in Section 3.

In [2]:
# Confirm the "available before prediction moment" claims above aren't just asserted:
# none of the 90d feature columns share a name pattern with the label-building 30d windows.
feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
overlap_with_30d_windows = [c for c in feature_cols if "last_30d" in c or "prev_30d" in c]
print("Feature columns that are actually 30d-window (label-adjacent) columns:", overlap_with_30d_windows)
print("-> empty list confirms the feature set only uses the 90d window, never the label's own 30d windows.")

print("\nTier column value counts (categorical features, checked for the 'unknown' fallback actually firing):")
for col in ["freshness_tier", "word_count_tier"]:
    print(f"  {col}:", df[col].fillna("unknown").value_counts().to_dict())

Feature columns that are actually 30d-window (label-adjacent) columns: []
-> empty list confirms the feature set only uses the 90d window, never the label's own 30d windows.

Tier column value counts (categorical features, checked for the 'unknown' fallback actually firing):
  freshness_tier: {'0-30': 20480, '91-180': 9171, '31-90': 175, '181+': 174}
  word_count_tier: {'2000-3500': 11263, 'unknown': 7699, '3500+': 6285, '1000-2000': 3780, '<1000': 973}


## 3. The leakage hunt

Attack test: add the six last-30d/prev-30d columns -- the label's own numerator and denominator
-- back into the "clean" feature set, and confirm the score visibly jumps. If it doesn't move,
the test itself is broken and none of the "clean" numbers elsewhere can be trusted either.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

groups = df["client_id"]
RANDOM_STATE = 42
gs = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gs.split(X, y, groups))

def score(X_variant, label):
    pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
    pipe.fit(X_variant.iloc[train_idx], y.iloc[train_idx])
    proba = pipe.predict_proba(X_variant.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[test_idx], proba)
    print(f"{label}: ROC-AUC = {auc:.3f}")
    return auc

auc_clean = score(X, "WITHOUT suspects (the feature set built in Section 1)")

leaky_cols = ["impressions_last_30d", "impressions_prev_30d",
              "clicks_last_30d", "clicks_prev_30d",
              "sessions_last_30d", "sessions_prev_30d"]
leaky_frame = df[leaky_cols].apply(pd.to_numeric, errors="coerce").fillna(0).reset_index(drop=True)
X_leaky = pd.concat([X, leaky_frame], axis=1)
auc_leaky = score(X_leaky, "WITH suspects added back (trend_pct's own numerator/denominator)")

print(f"\nCollapse test: {auc_leaky:.3f} -> {auc_clean:.3f}, a {auc_leaky - auc_clean:.3f} point drop.")
print("The jump confirms these six columns leak the label and must stay excluded.")

WITHOUT suspects (the feature set built in Section 1): ROC-AUC = 0.583


WITH suspects added back (trend_pct's own numerator/denominator): ROC-AUC = 0.848

Collapse test: 0.848 -> 0.583, a 0.265 point drop.
The jump confirms these six columns leak the label and must stay excluded.


## 4. What I excluded and why

| Excluded field(s) | Why |
|---|---|
| `trend_direction`, `trend_pct` | These **define** the label itself -- a feature can never also be its own label. |
| `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d` | The label's own numerator and denominator. Confirmed leaking above: adding them back inflates ROC-AUC. |
| `client_id`, `content_id` | Pseudonymous identifiers -- used only to group the train/test split, never as model inputs (a client ID predicting decline would just mean the model memorized which client is which). |
| `provider_used`, `model_used` | Metadata about *how* the content was produced, not about its performance -- including it risks the model learning a production-tooling artifact instead of a genuine content signal, and it's a thin, unevenly distributed field. |
| `age_tier_order` | A pure re-encoding of `age_tier`/`content_age_days` already in the feature set -- redundant, not leaky, dropped to avoid a duplicated signal getting double-weighted. |

In [4]:
print("Excluded columns actually still present in the raw file (never fed to the model):")
excluded = ["trend_direction", "trend_pct",
            "impressions_last_30d", "impressions_prev_30d",
            "clicks_last_30d", "clicks_prev_30d",
            "sessions_last_30d", "sessions_prev_30d",
            "client_id", "content_id",
            "provider_used", "model_used",
            "age_tier_order"]
present = [c for c in excluded if c in df.columns]
print(present)
print(f"\n{len(present)} of {len(excluded)} excluded fields confirmed present in raw data but absent from X.")
print(f"Columns actually in the feature matrix X: {X.shape[1]}")

Excluded columns actually still present in the raw file (never fed to the model):
['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'client_id', 'content_id', 'provider_used', 'model_used', 'age_tier_order']

13 of 13 excluded fields confirmed present in raw data but absent from X.
Columns actually in the feature matrix X: 59


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are pseudonyms already shipped in the dataset)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.